In [ ]:
# SINGLE-CELL benchmark: TWO independent groups (FULL vs GOLD, SMALL vs small GOLD)
# - Metrics per feature per file: F1/Recall/Precision/Accuracy + McNemar (exact + chi2 w/ CC)
# - Summary table: mean metrics per file across all features (unweighted mean)
# - Saves CSVs for each group + combined summaries

# ----------------------------
# 0) Auto-install dependencies
# ----------------------------
import sys, subprocess, importlib, os, re
def _ensure(pkg, import_name=None):
    try:
        importlib.import_module(import_name or pkg)
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

_ensure("pandas")
_ensure("numpy")
_ensure("scikit-learn", "sklearn")
_ensure("statsmodels")
_ensure("scipy")

# ----------------------------
# 1) Imports
# ----------------------------
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score
from statsmodels.stats.contingency_tables import mcnemar

# ----------------------------
# 2) Config
# ----------------------------
FEATURES = [
    "DirectAddress","Attachment","SelfDisclose","ReplySeeking",
    "Backseat", "Dimension", "CommRitual","Monetary", "Emotes"
    
]

GROUPS = [
    {
        "group_name": "FULL",
        "gold_file": "annotated_messages_Full_GOLD.xlsx",
        "models": {
            "1_41_Simple_NoContext": "1_41_Simple_NoContext.xlsx",
            "2_41_Enhanced_NoContext": "2_41_Enhanced_NoContext.xlsx",
            "10_51_Enhanced_NoContext": "10_51_Enhanced_NoContext.xlsx",
            "12_51_Simple_NoContext": "12_51_Simple_NoContext.xlsx",
            "14_51_Enhanced_NoContext_CCOT": "14_51_Enhanced_NoContext_CCOT.xlsx",
            "15_41_CCOT_NoContext": "15_41_CCOT_NoContext.xlsx",
            "18_51_Zero_NoContext": "18_51_Zero_NoContext.xlsx",
            "19_51_CoT_NoContext": "19_51_CoT_NoContext.xlsx",
            "20_51_ToT_NoContext": "20_51_ToT_NoContext.xlsx",
            "proposed": "proposed.xlsx",
            "PPFixedK70": "PPFixedK70.xlsx",
            "PPFixedK5": "PPFixedK5.xlsx",
            "PPCCOT": "PPCOT.xlsx",
            "PPmini": "PPmini.xlsx",
            "LiaHR": "24_LiaHR.xlsx",
            "EEStyle": "25_EEStyle.xlsx",
            "AnnoLLM": "23_AnnoLLM.xlsx",

        },
        "out_prefix": "benchmark_full"
    },
    {
        "group_name": "SMALL",
        "gold_file": "small_messages_gold.xlsx",
        "models": {
            "5_41_Enhanced_NoContext_Small": "5_41_Enhanced_NoContext_Small.xlsx",
            "6_41FT_Enhanced_NoContext_Small": "6_41FT_Enhanced_NoContext_Small.xlsx",  # will also try without .xlsx if missing
            "7_51_Enhanced_NoContext_Small": "7_51_Enhanced_NoContext_Small.xlsx",
        },
        "out_prefix": "benchmark_small"
    }
]

# ----------------------------
# 3) Helper functions
# ----------------------------
def _norm_colname(c: str) -> str:
    return re.sub(r"\s+", "", str(c)).strip().lower()

def _find_matching_column(df: pd.DataFrame, target: str) -> str:
    t = _norm_colname(target)
    for c in df.columns:
        if _norm_colname(c) == t:
            return c
    raise KeyError(f"Column '{target}' not found. Available columns: {list(df.columns)[:30]}{'...' if len(df.columns)>30 else ''}")

def _to_binary(s: pd.Series) -> pd.Series:
    # Treat only "Y" as 1; everything else (including empty) as 0
    s = s.astype("string").fillna("").str.strip().str.upper()
    return (s == "Y").astype(int)

def _to_dimension_binary(s: pd.Series) -> pd.Series:
    # P => 1, N => 0, otherwise NaN
    s = s.astype("string").fillna("").str.strip().str.upper()
    return pd.Series(
        np.where(s.isin(["P","POS","POSITIVE"]), 1,
                 np.where(s.isin(["N","NEG","NEGATIVE"]), 0, np.nan)),
        index=s.index
    )

def _mcnemar_table(y_gold: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    a = np.sum((y_gold==0)&(y_pred==0))
    b = np.sum((y_gold==0)&(y_pred==1))
    c = np.sum((y_gold==1)&(y_pred==0))
    d = np.sum((y_gold==1)&(y_pred==1))
    return np.array([[a,b],[c,d]])

def _resolve_path(path: str) -> str:
    # Best-effort: try exact, then add/remove .xlsx
    if os.path.exists(path):
        return path
    if not path.lower().endswith(".xlsx") and os.path.exists(path + ".xlsx"):
        return path + ".xlsx"
    if path.lower().endswith(".xlsx"):
        alt = path[:-5]
        if os.path.exists(alt):
            return alt
    raise FileNotFoundError(f"File not found: {path}")

def _evaluate_group(group_cfg: dict) -> tuple[pd.DataFrame, pd.DataFrame]:
    gname = group_cfg["group_name"]
    gold_path = _resolve_path(group_cfg["gold_file"])
    model_map = {k: _resolve_path(v) for k,v in group_cfg["models"].items()}

    gold_df = pd.read_excel(gold_path)
    model_dfs = {k: pd.read_excel(p) for k,p in model_map.items()}

    # Validate shapes (rows must align within group)
    shapes = {"GOLD": gold_df.shape, **{k: df.shape for k,df in model_dfs.items()}}
    if len(set(shapes.values())) != 1:
        raise ValueError(f"[{gname}] Shape mismatch across files: {shapes}")

    # Column mappings per dataframe
    gold_colmap = {c: _find_matching_column(gold_df, c) for c in FEATURES}
    model_colmaps = {name: {c: _find_matching_column(df, c) for c in FEATURES} for name, df in model_dfs.items()}

    rows = []
    for feat in FEATURES:
        gcol = gold_colmap[feat]
        y_gold_full = _to_binary(gold_df[gcol]) if feat != "Dimension" else _to_dimension_binary(gold_df[gcol])

        for model_name, dfm in model_dfs.items():
            mcol = model_colmaps[model_name][feat]
            y_pred_full = _to_binary(dfm[mcol]) if feat != "Dimension" else _to_dimension_binary(dfm[mcol])

            # For Dimension, drop rows with invalid values; for other features, values are always 0/1 after _to_binary
            mask = (~pd.isna(y_gold_full)) & (~pd.isna(y_pred_full))
            y_gold = y_gold_full[mask].astype(int).to_numpy()
            y_pred = y_pred_full[mask].astype(int).to_numpy()

            f1  = f1_score(y_gold, y_pred, zero_division=0)
            rec = recall_score(y_gold, y_pred, zero_division=0)
            prec= precision_score(y_gold, y_pred, zero_division=0)
            acc = accuracy_score(y_gold, y_pred)

            table = _mcnemar_table(y_gold, y_pred)

            # McNemar: requires discordant pairs; statsmodels handles b+c=0 but p-values become 1.0 typically
            p_exact = mcnemar(table, exact=True).pvalue
            p_chi2_cc = mcnemar(table, exact=False, correction=True).pvalue

            rows.append({
                "Group": gname,
                "Feature": feat,
                "Model": model_name,
                "N": int(len(y_gold)),
                "F1": float(f1),
                "Recall": float(rec),
                "Precision": float(prec),
                "Accuracy": float(acc),
                "McNemar_p_exact": float(p_exact),
            })

    metrics_long = pd.DataFrame(rows)

    # Mean metrics across features for each model (unweighted mean over FEATURES)
    mean_summary = (
        metrics_long
        .groupby(["Group","Model"], as_index=False)[["F1","Recall","Precision","Accuracy"]]
        .mean()
        .rename(columns={
            "F1":"Mean_F1",
            "Recall":"Mean_Recall",
            "Precision":"Mean_Precision",
            "Accuracy":"Mean_Accuracy"
        })
        .sort_values(["Group","Mean_F1"], ascending=[True, False])
        .reset_index(drop=True)
    )

    return metrics_long, mean_summary

# ----------------------------
# 4) Run both groups + save
# ----------------------------
all_long = []
all_means = []

for g in GROUPS:
    long_df, mean_df = _evaluate_group(g)
    all_long.append(long_df)
    all_means.append(mean_df)

    out_prefix = g["out_prefix"]
    long_path = f"{out_prefix}_metrics_long.csv"
    mean_path = f"{out_prefix}_mean_metrics_by_file.csv"
    long_df.to_csv(long_path, index=False)
    mean_df.to_csv(mean_path, index=False)

    print(f"[{g['group_name']}] Saved: {long_path}, {mean_path}")

metrics_long_all = pd.concat(all_long, ignore_index=True)
means_all = pd.concat(all_means, ignore_index=True)

metrics_long_all.to_csv("benchmark_ALL_groups_metrics_long.csv", index=False)
means_all.to_csv("benchmark_ALL_groups_mean_metrics_by_file.csv", index=False)
print("Saved: benchmark_ALL_groups_metrics_long.csv, benchmark_ALL_groups_mean_metrics_by_file.csv")

# ----------------------------
# 5) Display
# ----------------------------
pd.set_option("display.max_columns", 200)
display(metrics_long_all.sort_values(["Group","Feature","Model"]).reset_index(drop=True))
display(means_all)

print(
    "\nMcNemar quick read:\n"
    "- The 2x2 table is [[a,b],[c,d]] where b and c are the discordant counts.\n"
    "- If p < 0.05: evidence that the model’s positive rate differs from GOLD (systematic over/under-calling).\n"
    "- If p >= 0.05: no evidence of marginal-rate difference; errors may still exist but are more balanced."
)
